# Multivariate Questions\n\n1. How do age, experience (prior fights), and prior win rate relate to winning *together*?\n2. Which division is easiest to become champion in?\n3. (Own question) Does a reach advantage matter more in some weight divisions than others?\n\nAs with the bivariate notebook, questions 1 and 3 use the **individual-level** reshape (one row per fighter-per-fight, `won` as the outcome for that fighter directly) so the known red-corner recording bias (`docs/results.md`) can't leak into the answer - corner is never part of the comparison."

In [ ]:
import sys
sys.path.insert(0, "../src/data")
sys.path.insert(0, "../src/features")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cleaning import clean_fights
from engineering import add_matchup_features, add_fighter_history_features

master = pd.read_csv("../data/master.csv")
fights = clean_fights(master)
fights = add_matchup_features(fights)
fights = add_fighter_history_features(fights)

r_side = fights[["fight_id", "r_fighter_id", "r_age", "r_prior_fights", "r_prior_win_rate",
                  "r_reach_inches", "b_reach_inches", "weight_class", "winner_id"]].rename(columns={
    "r_fighter_id": "fighter_id", "r_age": "age", "r_prior_fights": "prior_fights",
    "r_prior_win_rate": "prior_win_rate", "r_reach_inches": "own_reach", "b_reach_inches": "opp_reach"})
b_side = fights[["fight_id", "b_fighter_id", "b_age", "b_prior_fights", "b_prior_win_rate",
                  "b_reach_inches", "r_reach_inches", "weight_class", "winner_id"]].rename(columns={
    "b_fighter_id": "fighter_id", "b_age": "age", "b_prior_fights": "prior_fights",
    "b_prior_win_rate": "prior_win_rate", "b_reach_inches": "own_reach", "r_reach_inches": "opp_reach"})
individual = pd.concat([r_side, b_side])
individual["won"] = (individual["fighter_id"] == individual["winner_id"]).astype(int)
individual["reach_advantage"] = individual["own_reach"] - individual["opp_reach"]

individual.shape

## 1. Age + Experience + Prior Win Rate -> Winning\n\nThree variables at once needs a different chart than a single bar - here that's a heatmap of age x prior win rate, **faceted into one panel per experience level** so all three show up together. Debut fighters (0 prior fights) have no prior win rate to speak of, so they're excluded here, same as in the bivariate layoff question.\n\nCells built from fewer than 30 fighters are grayed out and labeled instead of colored - a small cell can show a dramatic-looking rate (e.g. 0.75 from 4 wins out of 4 fights) that isn't reliable enough to read into."

In [ ]:
MIN_CELL_N = 30

sub = individual[individual["prior_fights"] >= 1].copy()
sub["age_bucket"] = pd.cut(sub["age"], bins=[16, 27, 33, 60], labels=["<28", "28-33", "34+"])
sub["form_bucket"] = pd.cut(sub["prior_win_rate"], bins=[-0.01, 0.4, 0.6, 1.01], labels=["<40%", "40-60%", ">60%"])
sub["exp_bucket"] = pd.cut(sub["prior_fights"], bins=[0, 5, 15, 100], labels=["1-5", "6-15", "16+"])

age_order = ["<28", "28-33", "34+"]
form_order = ["<40%", "40-60%", ">60%"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, exp in zip(axes, ["1-5", "6-15", "16+"]):
    part = sub[sub["exp_bucket"] == exp]
    rate = part.pivot_table(index="age_bucket", columns="form_bucket", values="won", aggfunc="mean", observed=True)
    n = part.pivot_table(index="age_bucket", columns="form_bucket", values="won", aggfunc="count", observed=True)
    rate = rate.reindex(index=age_order, columns=form_order)
    n = n.reindex(index=age_order, columns=form_order)
    masked = rate.where(n >= MIN_CELL_N)

    im = ax.imshow(masked, vmin=0.2, vmax=0.7, cmap="RdYlGn")
    ax.set_xticks(range(3))
    ax.set_xticklabels(form_order)
    ax.set_yticks(range(3))
    ax.set_yticklabels(age_order)
    ax.set_xlabel("prior win rate")
    ax.set_title(f"{exp} prior fights")

    for i in range(3):
        for j in range(3):
            count = n.values[i, j]
            if np.isnan(count):
                continue
            if count >= MIN_CELL_N:
                label = f"{rate.values[i, j]:.2f}\n(n={int(count)})"
            else:
                label = f"n={int(count)}\n(too small)"
            ax.text(j, i, label, ha="center", va="center", fontsize=8)

axes[0].set_ylabel("age")
fig.colorbar(im, ax=axes, shrink=0.7, label="win rate")
plt.show()

## 2. Which Division Is Easiest to Become Champion In?\n\n\"Easiest\" splits into two different real questions, so both get answered rather than picking one silently:\n\n- **Title turnover** - how many distinct fighters have won a title fight in a division, relative to the total number of title fights contested there. Low turnover means one or two fighters won nearly every one (a dominant, hard-to-dethrone champion); high turnover means many different fighters got to hold the belt.\n- **Path to a shot** - among fighters who got a title fight, how many pro fights did it typically take them to get there? A shorter path means it's easier to *reach* contention, even if the champion then holds on for a while.\n\n`weight_class` has 154 raw values, and many are defunct promotions (WEC, Strikeforce, Pride) with only a handful of title fights each - not a fair comparison against modern UFC divisions with 20-57 title fights of history. Filtering to `total_title_fights >= 13` turns out to cut the data at exactly the boundary between the 11 real, currently-standing UFC divisions and the legacy promotion belts (the largest of those tops out at 12)."

In [ ]:
title = fights[fights["title_fight"]]

div_stats = title.groupby("weight_class", observed=True).agg(
    total_title_fights=("winner_id", "size"), distinct_champs=("winner_id", "nunique"))
div_stats = div_stats[div_stats["total_title_fights"] >= 13]
div_stats["turnover_ratio"] = div_stats["distinct_champs"] / div_stats["total_title_fights"]
div_stats = div_stats.sort_values("turnover_ratio", ascending=False)

r_side_t = fights[["fight_id", "r_fighter_id", "r_prior_fights", "weight_class", "title_fight"]].rename(
    columns={"r_fighter_id": "fighter_id", "r_prior_fights": "prior_fights"})
b_side_t = fights[["fight_id", "b_fighter_id", "b_prior_fights", "weight_class", "title_fight"]].rename(
    columns={"b_fighter_id": "fighter_id", "b_prior_fights": "prior_fights"})
title_long = pd.concat([r_side_t, b_side_t])
title_long = title_long[title_long["title_fight"] & title_long["weight_class"].isin(div_stats.index)]
avg_path = title_long.groupby("weight_class", observed=True)["prior_fights"].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(div_stats.index, div_stats["turnover_ratio"], color="teal")
axes[0].invert_yaxis()
axes[0].set_xlabel("distinct champions / total title fights")
axes[0].set_title("Title Turnover (higher = easier)")

axes[1].barh(avg_path.index, avg_path.values, color="tab:brown")
axes[1].invert_yaxis()
axes[1].set_xlabel("avg prior fights before a title shot")
axes[1].set_title("Path to a Title Shot (lower = easier)")
plt.tight_layout()
plt.show()

div_stats, avg_path

## 3. (Own Question) Does a Reach Advantage Matter More in Some Divisions?\n\n`height_diff`/`reach_diff` showed up as real but modest signals back in `eda.ipynb`, computed the usual red-minus-blue way. The natural follow-up: does reach matter the same amount everywhere, or does it interact with weight class? Heavier divisions have more knockout power and less scrambling speed to close distance, so reach might matter far more there than in a fast, high-output division like Lightweight.\n\nUsing `reach_advantage = own_reach - opponent_reach` per fighter (not per corner) for the same bias-avoidance reason as everywhere else in this notebook, restricted to the 8 standard men's divisions with reach data (women's divisions have too many `NaN` reach values to bucket reliably here)."

In [ ]:
main_divisions = ["Flyweight", "Bantamweight", "Featherweight", "Lightweight",
                   "Welterweight", "Middleweight", "Light Heavyweight", "Heavyweight"]

sub3 = individual[individual["weight_class"].isin(main_divisions)].copy()
sub3["reach_bucket"] = pd.cut(sub3["reach_advantage"], bins=[-100, -2, 2, 100],
                                labels=["shorter (-2\" or more)", "even (-1\" to 1\")", "longer (+2\" or more)"])

pivot = sub3.pivot_table(index="weight_class", columns="reach_bucket", values="won", aggfunc="mean", observed=True)
pivot = pivot.reindex(main_divisions)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(pivot))
width = 0.25
for i, bucket in enumerate(pivot.columns):
    ax.bar(x + i * width, pivot[bucket], width, label=str(bucket))
ax.axhline(0.5, color="gray", linestyle="--")
ax.set_xticks(x + width)
ax.set_xticklabels(pivot.index, rotation=45, ha="right")
ax.set_ylabel("win rate")
ax.set_title("Win Rate by Reach Advantage, per Division")
ax.legend()
plt.tight_layout()
plt.show()

pivot